# Polarimetric SAR Reconstruction Pipeline Evaluation

This notebook provides a comprehensive evaluation of various aspects of the polarimetric SAR reconstruction pipeline. We systematically compare:

- **Layer Modes**: Complex vs Real (with different pipeline types)
- **Model Architectures**: AutoEncoder vs LatentAutoEncoder
- **🔬 Latent Vector Size Impact**: Systematic study of compression vs quality trade-offs (16D, 32D, 64D, 128D, 256D, 512D)
- **Transform Configurations**: Different preprocessing and augmentation strategies
- **Training Configurations**: Various hyperparameter settings

The notebook demonstrates the integrated registry system for models, transforms, and metrics that we've developed for standardized and reproducible experiments.

### 🎯 Key Research Focus: Latent Space Dimensionality

A major focus of this evaluation is understanding how the latent vector size in LatentAutoEncoder affects:
- **Reconstruction Quality**: Trade-off between compression and fidelity
- **Model Efficiency**: Parameter count and computational requirements
- **Training Dynamics**: Convergence speed and stability
- **Generalization**: Performance on unseen data

## Parameters and Experiment Configuration

Define notebook parameters for reproducibility and Papermill compatibility. These parameters control the scope and scale of our pipeline evaluation experiments.

In [ ]:
base_config = {
    # Training configuration
    "mode": "full",
    "world_size": 4,
    
    # Optimizer configuration
    "optim": {
        "algo": "AdamW",
        "params": {
            "lr": 0.0005,  # Reduced from 0.0005 for more stable training
            "weight_decay": 0.001,  # Increased weight decay for better regularization
            "betas": [0.9, 0.999]
        }
    },
    
    # Scheduler configuration with ReduceLROnPlateau for adaptive learning
    "scheduler": {
        "name": "ReduceLROnPlateau",
        "params": {
            "mode": "min",
            "factor": 0.5,
            "patience": 5,
            "min_lr": 0.000001
        }
    },
    
    # Warmup configuration
    "warmup": {
        "epochs": 5
    },
    
    # Logging configuration
    "logging": {
        "logdir": "./logs",
        "wandb": {
            "entity": "complex-gen-ai-sar",
            "project": "polsar_reconstruction_sf_alos2",
            "name": "autoencoder"
        }
    },
    
    # Loss configuration - using new generic naming
    "loss": {
        "name": "MSE"
    },
    
    "task": "reconstruction",
    
    # Data configuration specific to reconstruction
    "data": {
        "batch_size": 64,
        "crop": {
            "end_col": 3200,
            "end_row": 7500,
            "start_col": 1000,
            "start_row": 3000
        },
        "dataset": {
            "name": "ALOSDataset",
            "trainpath": "../datasets/SAN_FRANCISCO_ALOS2"
        },
        "num_workers": 4,
        "patch_size": 64,
        "patch_stride": 64,
        "test_ratio": 0.1,
        "valid_ratio": 0.1
    },
    
    # Model configuration specific to reconstruction
    "model": {
        "activation": "modReLU",
        "channels_width": 8,
        "class": "AutoEncoder",
        "layer_mode": "complex",
        "num_layers": 2,
        "normalization_layer": "batch",
        "downsampling_layer": None,
        "upsampling_layer": "transpose",
        "residual": True,
        "dropout": 0.1
    }
}


In [ ]:
# Parameters cell for Papermill
tags = ['parameters']

# Experiment control parameters
run_quick_demo = False  # Set to False for full evaluation
max_epochs_demo = 2    # For quick demo runs
max_epochs_full = 50   # For full evaluation

# Sensitivity Analysis Parameters
enable_sensitivity_analysis = True  # Enable/disable sensitivity analysis
sensitivity_n_runs = 3  # Number of runs per experiment for sensitivity analysis
sensitivity_different_seeds = True  # Use different random seeds for each run
sensitivity_base_seed = 42  # Base seed for reproducible sensitivity analysis

# Experiment scope control
evaluate_layer_modes_ae = False      # Compare complex vs real modes for AutoEncoder
evaluate_layer_modes_latent_ae = False  # Compare complex vs real modes for LatentAutoEncoder
evaluate_latent_size = False  # Compare different latent vector sizes
evaluate_transforms = True       # Compare transform configurations
evaluate_hyperparams = False     # Compare different hyperparameters

# Results and logging
results_dir = 'logs/pipeline_evaluation'
save_detailed_results = True
enable_wandb = True  # Set to True to log to Weights & Biases

# Reproducibility
fixed_seed = 42

## Setup and Imports

Import all required libraries and local utilities. This includes our integrated registry system for models, transforms, and metrics.

In [ ]:
# coding: utf-8
%matplotlib inline

# Standard library imports
import os
import sys
import pathlib
import copy
import json
import time
from datetime import datetime
from typing import Dict, List, Any, Tuple

# Display and visualization
from IPython.display import display, HTML
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Third-party imports
import numpy as np
import yaml
import torch
from tqdm.auto import tqdm

# Optional imports
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("W&B not available, skipping wandb integration")

# Add project root to Python path (notebook-safe version)
# For notebooks, we need to determine the project root differently since __file__ is not available
current_dir = pathlib.Path.cwd()

# Check if we're in the notebooks directory, and if so, go up one level to project root
if current_dir.name == "notebooks":
    project_root = current_dir.parent
elif (current_dir / "notebooks").exists():
    # We're already at project root
    project_root = current_dir
else:
    # Try to find project root by looking for characteristic files
    project_root = current_dir
    for parent in [current_dir] + list(current_dir.parents):
        if (parent / "pyproject.toml").exists() or (parent / "src").exists():
            project_root = parent
            break

sys.path.insert(0, str(project_root))

# Ensure src is also in path
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {project_root}")
print(f"Source path: {src_path}")
print(f"Python path includes: {[p for p in sys.path[:3]]}")

# Add src to path for imports (backup method)
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Local imports - Core utilities
from cvnn.utils import setup_logging, set_seed

# Local imports - Registry system (our integrated system)
from cvnn.transform_registry import list_available_transforms

# Set up logging
logger = setup_logging(__name__)

print("✅ All imports successful!")
print(f"📊 PyTorch version: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")
print(f"📈 W&B available: {WANDB_AVAILABLE}")

## Environment & Reproducibility

Set up reproducible environment, random seeds, and logging configuration for consistent experiments across different pipeline configurations.

In [ ]:
# Set up reproducibility
set_seed(fixed_seed)
print(f"🎯 Fixed seed set to: {fixed_seed}")

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = torch.cuda.is_available()
print(f"🖥️  Using device: {device}")

# Create results directory
results_path = pathlib.Path(results_dir)
results_path.mkdir(parents=True, exist_ok=True)
print(f"📁 Results will be saved to: {results_path}")

# Log environment info
print(f"\n📋 Environment Info:")
print(f"   Python: {sys.version.split()[0]}")
print(f"   NumPy: {np.__version__}")
print(f"   PyTorch: {torch.__version__}")

# Log Git commit if available
try:
    import subprocess
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"]).strip().decode()
    print(f"   Git SHA: {git_sha[:8]}")
except:
    print("   Git SHA: Not available")

# Configure matplotlib for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("\n✅ Environment setup complete!")

## Experiment Grid Definition

Define systematic experiment configurations to evaluate different aspects of the pipeline. Each experiment will test specific combinations of:

1. **Layer Modes & Pipelines**: Complex vs Real processing with different pipeline types
2. **Model Architectures**: AutoEncoder vs LatentAutoEncoder  
3. **Latent Vector Sizes**: Impact of latent dimensionality on compression and reconstruction quality (16D, 32D, 64D, 128D, 256D, 512D)
4. **Transform Configurations**: Different preprocessing and augmentation strategies
5. **Hyperparameters**: Various training configurations

### Key Research Questions

- **Compression vs Quality Trade-off**: How does latent vector size affect reconstruction quality?
- **Parameter Efficiency**: What's the relationship between latent size and model parameters?
- **Convergence Behavior**: Do smaller latent spaces converge faster or slower?
- **Generalization**: Which latent sizes generalize better to unseen data?

In [ ]:
# Define the experiment grid - using pipeline-based configuration approach
experiment_configs = []

# 1. Layer Mode Experiments for AutoEncoder - Compare processing approaches
if evaluate_layer_modes_ae:
    layer_mode_experiments_ae = [
        {
            "name": "Complex_Native_AutoEncoder",
            "description": "Native complex processing",
            "config_updates": {
                "model": {"layer_mode": "complex", 
                          "class": "AutoEncoder"},
                "data": {
                    "real_pipeline_type": None
                }
            }
        },
        {
            "name": "Real_DualChannel_AutoEncoder",
            "description": "Real processing with real/imaginary channels",
            "config_updates": {
                "model": {"layer_mode": "real",
                          "class": "AutoEncoder"},
                "data": {"real_pipeline_type": "complex_dual_real"}
            }
        }        
    ]
    experiment_configs.extend(layer_mode_experiments_ae)

# 2. Layer Mode Experiments for LatentAutoEncoder - Compare processing approaches 
if evaluate_layer_modes_latent_ae:
    layer_mode_experiments_latent_ae = [
        {
            "name": "Complex_Native_LatentAutoEncoder",
            "description": "Native complex processing",
            "config_updates": {
                "model": {"layer_mode": "complex", 
                          "class": "LatentAutoEncoder",
                          "latent_dim": 128},
                "data": {
                    "real_pipeline_type": None
                }
            }
        },
        {
            "name": "Real_DualChannel_LatentAutoEncoder",
            "description": "Real processing with real/imaginary channels",
            "config_updates": {
                "model": {"layer_mode": "real",
                          "class": "LatentAutoEncoder",
                          "latent_dim": 128},
                "data": {"real_pipeline_type": "complex_dual_real"}
            }
        }   
    ]
    experiment_configs.extend(layer_mode_experiments_latent_ae)

# 3. Latent Vector Size Experiments - Study impact of latent dimensionality
if evaluate_latent_size:
    print("🔬 Adding latent vector size experiments for LatentAutoEncoder...")
    
    latent_size_experiments = [
        {
            "name": "LatentAE_Tiny",
            "description": "LatentAutoEncoder with very small latent space (16D)",
            "config_updates": {
                "model": {
                    "layer_mode": "complex",
                    "class": "LatentAutoEncoder",
                    "latent_dim": 16
                },
                "data": {
                    "real_pipeline_type": None
                }
            }
        },
        {
            "name": "LatentAE_Small",
            "description": "LatentAutoEncoder with small latent space (32D)",
            "config_updates": {
                "model": {
                    "layer_mode": "complex",
                    "class": "LatentAutoEncoder",
                    "latent_dim": 32
                },
                "data": {
                    "real_pipeline_type": None
                }
            }
        },
        {
            "name": "LatentAE_Medium",
            "description": "LatentAutoEncoder with medium latent space (64D) - baseline",
            "config_updates": {
                "model": {
                    "layer_mode": "complex",
                    "class": "LatentAutoEncoder",
                    "latent_dim": 64
                },
                "data": {
                    "real_pipeline_type": None
                }
            }
        },
        {
            "name": "LatentAE_Large",
            "description": "LatentAutoEncoder with large latent space (128D)",
            "config_updates": {
                "model": {
                    "layer_mode": "complex",
                    "class": "LatentAutoEncoder",
                    "latent_dim": 128
                },
                "data": {
                    "real_pipeline_type": None
                }
            }
        },
        {
            "name": "LatentAE_XLarge",
            "description": "LatentAutoEncoder with extra large latent space (256D)",
            "config_updates": {
                "model": {
                    "layer_mode": "complex",
                    "class": "LatentAutoEncoder",
                    "latent_dim": 256
                },
                "data": {
                    "real_pipeline_type": None
                }
            }
        },
        {
            "name": "LatentAE_Huge",
            "description": "LatentAutoEncoder with huge latent space (512D)",
            "config_updates": {
                "model": {
                    "layer_mode": "complex",
                    "class": "LatentAutoEncoder",
                    "latent_dim": 512
                },
                "data": {
                    "real_pipeline_type": None
                }
            }
        }
    ]
    experiment_configs.extend(latent_size_experiments)
    print(f"   Added {len(latent_size_experiments)} latent size experiments: {[exp['name'] for exp in latent_size_experiments]}")

# 4. Transform Configuration Experiments - Compare preprocessing
if evaluate_transforms:
    # Show available transforms for different layer modes (with error handling)
    print("📋 Available transforms by layer mode:")
    try:
        for mode in ["real", "complex"]:
            available = list_available_transforms(mode)
            print(f"   {mode}: {available}")
    except Exception as e:
        print(f"   ⚠️  Could not list available transforms: {e}")
        print("   Using basic transform configuration instead...")
    
    transform_experiments = [
        {
            "name": "Transforms_Basic",
            "description": "Basic transform pipeline",
            "config_updates": {
                "data": {
                    "transforms": []  # Only dataset base transforms
                }
            }
        },
        {
            "name": "Transforms_WithRandomHorizontalFlip",
            "description": "Transform pipeline with resizing",
            "config_updates": {
                "data": {
                    # Use list instead of tuple to avoid YAML serialization issues
                    "transforms": [
                        {"name": "randomhorizontalflip"}
                    ]
                }
            }
        }
    ]
    # Add some basic transforms for comparison
    experiment_configs.extend(transform_experiments)

# 5. Hyperparameter Experiments - Compare training configurations
if evaluate_hyperparams:
    hyperparam_experiments = [
        {
            "name": "HyperParam_Baseline",
            "description": "Baseline hyperparameters",
            "config_updates": {
            }
        },
        {
            "name": "HyperParam_Layer_3_Width_32",
            "description": "Deeper network with more capacity",
            "config_updates": {
                "model": {
                    "num_layers": 3,
                    "channels_ratio": 32
                }
            }
        }
    ]
    experiment_configs.extend(hyperparam_experiments)

# Display experiment summary
print(f"🧪 Defined {len(experiment_configs)} experiments:")
for i, exp in enumerate(experiment_configs, 1):
    print(f"   {i:2d}. {exp['name']:25} - {exp['description']}")

# Adjust for demo mode
if run_quick_demo:
    # For demo, run only a subset
    experiment_configs = experiment_configs[:3]  # First 3 experiments
    print(f"\n🏃‍♀️ Running in quick demo mode with {len(experiment_configs)} experiments.")
else:
    print("\n🚀 Running in full evaluation mode.")

# Set epochs based on mode
if run_quick_demo:
    base_config["nepochs"] = max_epochs_demo
else:
    base_config["nepochs"] = max_epochs_full
    
print(f"🗓️  Will train for {base_config['nepochs']} epochs per experiment.")

In [ ]:
from cvnn.experiments import run_experiment
import tempfile
import os
import numpy as np
from scipy import stats

def run_pipeline_experiment(exp_config: Dict[str, Any], exp_index: int, total_experiments: int, run_id: int = 0, seed_offset: int = 0) -> Dict[str, Any]:
    """
    Run a single experiment using the standard reconstruction pipeline.
    
    This function uses the project's standard run_experiment function with configuration variants,
    ensuring consistency with the main project workflow.
    
    Args:
        exp_config: Experiment configuration dictionary
        exp_index: Current experiment index (0-based)
        total_experiments: Total number of experiments
        run_id: Run identifier for sensitivity analysis (0-based)
        seed_offset: Offset to add to the base seed for this run
        
    Returns:
        Dictionary with experiment results and metadata
    """
    exp_name = exp_config["name"]
    exp_desc = exp_config["description"]
    
    if enable_sensitivity_analysis and sensitivity_n_runs > 1:
        print(f"\n🧪 Experiment {exp_index + 1}/{total_experiments} - Run {run_id + 1}/{sensitivity_n_runs}: {exp_name}")
    else:
        print(f"\n🧪 Experiment {exp_index + 1}/{total_experiments}: {exp_name}")
    print(f"   Description: {exp_desc}")
    print(f"   Started at: {datetime.now().strftime('%H:%M:%S')}")
    if seed_offset > 0:
        print(f"   Random seed: {fixed_seed + seed_offset}")
    
    start_time = time.time()
    
    # 1. Create experiment configuration by merging updates
    config = copy.deepcopy(base_config)

    # Apply configuration updates
    for section, updates in exp_config["config_updates"].items():
        if section not in config:
            config[section] = {}
        config[section].update(updates)
            
    # Clean up None values (especially for complex mode where real_pipeline_type should not be set)
    if config["model"]["layer_mode"] == "complex":
        if "real_pipeline_type" in config["data"]:
            del config["data"]["real_pipeline_type"]
    
    # Set epochs based on demo mode
    if run_quick_demo:
        config["nepochs"] = max_epochs_demo
    else:
        config["nepochs"] = max_epochs_full
    
    # Ensure task is set correctly
    config["task"] = "reconstruction"
    
    # Set mode to train for training experiments
    config["mode"] = "train"
    
    # Add experiment metadata
    config["experiment"] = {
        "name": exp_name,
        "description": exp_desc,
        "index": exp_index,
        "run_id": run_id,
        "seed_offset": seed_offset
    }
    
    # Set random seed for this run if sensitivity analysis is enabled
    if enable_sensitivity_analysis and sensitivity_different_seeds:
        config["seed"] = fixed_seed + seed_offset
    else:
        config["seed"] = fixed_seed
    
    print(f"   ⚙️  Configuration created with task: {config.get('task', 'unknown')}")
    print(f"   🎯 Layer mode: {config.get('model', {}).get('layer_mode', 'unknown')}")
    print(f"   🏗️  Model class: {config.get('model', {}).get('class', 'unknown')}")
    
    # 2. Create temporary config file for the pipeline
    with tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False) as f:
        yaml.dump(config, f, default_flow_style=False)
        temp_config_path = f.name
    
    print(f"   📝 Created temporary config: {temp_config_path}")
    
    # 3. Run the experiment using the standard pipeline
    print(f"   🚀 Running experiment via standard pipeline...")

    try:
        # Use the project's standard run_experiment function
        history, metrics, logdir, model = run_experiment(
            config_path=temp_config_path,
            resume_logdir=False,
            mode_override=None
        )
        
        print(f"   ✅ Pipeline execution completed")
        print(f"   📁 Results saved to: {logdir}")
        
        # Extract training history
        training_history = {}
        if history:
            if isinstance(history, dict):
                training_history = history
            else:
                # Convert history object to dict if needed
                training_history = {
                    "train_loss": getattr(history, 'train_loss', []),
                    "valid_loss": getattr(history, 'valid_loss', []),
                }
        
        # Calculate final losses
        final_train_loss = training_history.get("train_loss", [])[-1] if training_history.get("train_loss") else None
        final_val_loss = training_history.get("valid_loss", [])[-1] if training_history.get("valid_loss") else None
        best_val_loss = min(training_history.get("valid_loss", [])) if training_history.get("valid_loss") else None
        best_epoch = training_history.get("valid_loss", []).index(best_val_loss) if best_val_loss else None
        
        print(f"   📊 Final validation loss: {final_val_loss:.6f}" if final_val_loss else "   📊 No validation loss recorded")
        print(f"   📈 Evaluation metrics: {metrics}")
        
    except Exception as pipeline_error:
        print(f"   ❌ Pipeline execution failed: {pipeline_error}")
        raise pipeline_error
    
    finally:
        # Clean up temporary config file
        if os.path.exists(temp_config_path):
            os.unlink(temp_config_path)
    
    # 4. Compile results in standard format
    end_time = time.time()
    duration = end_time - start_time
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    pipeline_results = {
        # Experiment metadata
        "experiment_name": exp_name,
        "experiment_description": exp_desc,
        "experiment_index": exp_index,
        "run_id": run_id,
        "seed_offset": seed_offset,
        "duration_seconds": duration,
        "timestamp": datetime.now().isoformat(),
        
        # Configuration
        "config": config,
        "layer_mode": config.get("model", {}).get("layer_mode", "unknown"),
        "model_class": config.get("model", {}).get("class", "AutoEncoder"),
        
        # Model info
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        
        # Training results
        "final_train_loss": final_train_loss,
        "final_val_loss": final_val_loss,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "training_history": training_history,
        
        # Evaluation metrics
        "metrics": metrics,
        
        # Pipeline info
        "logdir": str(logdir),
        
        # Status
        "status": "completed",
        "error": None
    }
    
    print(f"   ✅ Experiment completed in {duration:.1f}s")
    
    return pipeline_results


def run_sensitivity_experiment(exp_config: Dict[str, Any], exp_index: int, total_experiments: int) -> List[Dict[str, Any]]:
    """
    Run an experiment multiple times for sensitivity analysis.
    
    Args:
        exp_config: Experiment configuration dictionary
        exp_index: Current experiment index (0-based)
        total_experiments: Total number of experiments
        
    Returns:
        List of results from all runs
    """
    if not enable_sensitivity_analysis or sensitivity_n_runs <= 1:
        # Run single experiment
        return [run_pipeline_experiment(exp_config, exp_index, total_experiments)]
    
    print(f"\n🔄 Sensitivity Analysis: Running {sensitivity_n_runs} times for {exp_config['name']}")
    
    all_runs = []
    
    for run_id in range(sensitivity_n_runs):
        # Calculate seed offset for this run
        seed_offset = run_id * 1000 if sensitivity_different_seeds else 0
        
        try:
            result = run_pipeline_experiment(exp_config, exp_index, total_experiments, run_id, seed_offset)
            all_runs.append(result)
        except Exception as e:
            print(f"   ❌ Run {run_id + 1} failed: {e}")
            # Create a failed result entry
            failed_result = {
                "experiment_name": exp_config["name"],
                "experiment_description": exp_config["description"],
                "experiment_index": exp_index,
                "run_id": run_id,
                "seed_offset": seed_offset,
                "status": "failed",
                "error": str(e),
                "timestamp": datetime.now().isoformat(),
            }
            all_runs.append(failed_result)
    
    # Print summary for this experiment
    successful_runs = [r for r in all_runs if r.get("status") == "completed"]
    if len(successful_runs) > 0:
        val_losses = [r["final_val_loss"] for r in successful_runs if r.get("final_val_loss") is not None]
        if val_losses:
            mean_loss = np.mean(val_losses)
            std_loss = np.std(val_losses)
            print(f"   📊 Sensitivity Summary: {len(successful_runs)}/{sensitivity_n_runs} successful runs")
            print(f"   📈 Validation Loss: {mean_loss:.6f} ± {std_loss:.6f}")
    
    return all_runs


def calculate_sensitivity_statistics(results: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Calculate statistical measures across multiple runs of the same experiment.
    
    Args:
        results: List of results from multiple runs of the same experiment
        
    Returns:
        Dictionary with statistical measures
    """
    successful_results = [r for r in results if r.get("status") == "completed"]
    
    if len(successful_results) == 0:
        return {
            "n_successful_runs": 0,
            "n_failed_runs": len(results),
            "success_rate": 0.0
        }
    
    stats_dict = {
        "n_successful_runs": len(successful_results),
        "n_failed_runs": len(results) - len(successful_results),
        "success_rate": len(successful_results) / len(results),
        "experiment_name": successful_results[0]["experiment_name"],
        "experiment_description": successful_results[0]["experiment_description"],
    }
    
    # Calculate statistics for numerical metrics
    numerical_fields = [
        "final_train_loss", "final_val_loss", "best_val_loss", "best_epoch",
        "total_parameters", "trainable_parameters", "duration_seconds"
    ]
    
    for field in numerical_fields:
        values = [r[field] for r in successful_results if r.get(field) is not None]
        if values:
            stats_dict[f"{field}_mean"] = np.mean(values)
            stats_dict[f"{field}_std"] = np.std(values)
            stats_dict[f"{field}_min"] = np.min(values)
            stats_dict[f"{field}_max"] = np.max(values)
            if len(values) > 1:
                # Calculate confidence interval (95%)
                confidence_level = 0.95
                degrees_freedom = len(values) - 1
                sample_mean = np.mean(values)
                sample_stderr = stats.sem(values)
                confidence_interval = stats.t.interval(confidence_level, degrees_freedom, sample_mean, sample_stderr)
                stats_dict[f"{field}_ci_lower"] = confidence_interval[0]
                stats_dict[f"{field}_ci_upper"] = confidence_interval[1]
    
    # Calculate statistics for metrics
    if len(successful_results) > 0 and "metrics" in successful_results[0]:
        
        def extract_metric_values(metric_dict, prefix=""):
            """Recursively extract metric values, flattening nested dictionaries."""
            values = {}
            for key, value in metric_dict.items():
                full_key = f"{prefix}_{key}" if prefix else key
                if isinstance(value, dict):
                    values.update(extract_metric_values(value, full_key))
                elif isinstance(value, (int, float)) and not isinstance(value, bool):
                    values[full_key] = value
            return values
        
        # Get all metric keys from first result
        all_metric_keys = set()
        for result in successful_results:
            if "metrics" in result:
                metric_values = extract_metric_values(result["metrics"])
                all_metric_keys.update(metric_values.keys())

        # Calculate statistics for each metric
        for metric_key in all_metric_keys:
            values = []
            for result in successful_results:
                if "metrics" in result:
                    metric_values = extract_metric_values(result["metrics"])
                    if metric_key in metric_values:
                        values.append(metric_values[metric_key])
            
            if values:
                stats_dict[f"metric_{metric_key}_mean"] = np.mean(values)
                stats_dict[f"metric_{metric_key}_std"] = np.std(values)
                stats_dict[f"metric_{metric_key}_min"] = np.min(values)
                stats_dict[f"metric_{metric_key}_max"] = np.max(values)
                if len(values) > 1:
                    confidence_level = 0.95
                    degrees_freedom = len(values) - 1
                    sample_mean = np.mean(values)
                    sample_stderr = stats.sem(values)
                    confidence_interval = stats.t.interval(confidence_level, degrees_freedom, sample_mean, sample_stderr)
                    stats_dict[f"metric_{metric_key}_ci_lower"] = confidence_interval[0]
                    stats_dict[f"metric_{metric_key}_ci_upper"] = confidence_interval[1]
    
    # Copy configuration details from first successful result
    if len(successful_results) > 0:
        first_result = successful_results[0]
        for key in ["layer_mode", "model_class"]:
            if key in first_result:
                stats_dict[key] = first_result[key]
        
        # Extract config details
        if "config" in first_result:
            config = first_result["config"]
            model_config = config.get("model", {})
            stats_dict["latent_dim"] = model_config.get("latent_dim", np.nan)
            stats_dict["num_layers"] = model_config.get("num_layers", np.nan)
            stats_dict["channels_ratio"] = model_config.get("channels_ratio", np.nan)
    
    return stats_dict


print("✅ Pipeline-based experiment runner with sensitivity analysis defined")
print("🔄 Ready to execute experiments with statistical analysis across multiple runs")

## Running All Experiments

Execute all defined experiments systematically. Each experiment will be run independently with its own configuration, and results will be collected for analysis.

In [ ]:
# Debug: Check experiment configuration before running
print(f"🔍 Debugging experiment configuration...")
config = copy.deepcopy(base_config)

# Initialize results collection
all_results = []  # Individual run results
sensitivity_stats = []  # Aggregated statistics per experiment
experiment_start_time = time.time()

print(f"🚀 Starting pipeline evaluation with {len(experiment_configs)} experiments")
if enable_sensitivity_analysis:
    total_runs = len(experiment_configs) * sensitivity_n_runs
    print(f"   🔄 Sensitivity Analysis: {sensitivity_n_runs} runs per experiment ({total_runs} total runs)")
    print(f"   🎲 Different seeds: {sensitivity_different_seeds}")
else:
    print(f"   🔄 Single run mode (sensitivity analysis disabled)")
print(f"   Demo mode: {run_quick_demo}")
print(f"   Max epochs: {max_epochs_demo if run_quick_demo else max_epochs_full}")
print(f"   Device: {device}")
print(f"   Results dir: {results_dir}")
print(f"   Using: Pipeline-based ReconstructionExperiment")


# Run all experiments using the sensitivity analysis approach
for i, exp_config in enumerate(experiment_configs):
    print(f"\n" + "="*80)
    
    # Run the experiment (potentially multiple times for sensitivity analysis)
    experiment_runs = run_sensitivity_experiment(exp_config, i, len(experiment_configs))
    
    # Add all individual runs to results
    all_results.extend(experiment_runs)
    
    # Calculate sensitivity statistics
    if enable_sensitivity_analysis and len(experiment_runs) > 1:
        experiment_stats = calculate_sensitivity_statistics(experiment_runs)
        sensitivity_stats.append(experiment_stats)
        
        # Print sensitivity summary
        if experiment_stats["n_successful_runs"] > 0:
            print(f"\n📊 Sensitivity Analysis Summary for {exp_config['name']}:")
            print(f"   Success Rate: {experiment_stats['success_rate']:.1%} ({experiment_stats['n_successful_runs']}/{len(experiment_runs)})")
            
            # Print key metrics with confidence intervals
            key_metrics = [
                ("final_val_loss", "Validation Loss"),
                ("best_val_loss", "Best Validation Loss"),
                ("duration_seconds", "Training Time (s)"),
            ]
            
            for metric_key, metric_name in key_metrics:
                if f"{metric_key}_mean" in experiment_stats:
                    mean_val = experiment_stats[f"{metric_key}_mean"]
                    std_val = experiment_stats[f"{metric_key}_std"]
                    ci_lower = experiment_stats.get(f"{metric_key}_ci_lower", None)
                    ci_upper = experiment_stats.get(f"{metric_key}_ci_upper", None)
                    
                    if ci_lower is not None and ci_upper is not None:
                        print(f"   {metric_name}: {mean_val:.6f} ± {std_val:.6f} (95% CI: [{ci_lower:.6f}, {ci_upper:.6f}])")
                    else:
                        print(f"   {metric_name}: {mean_val:.6f} ± {std_val:.6f}")
    else:
        # For single runs, just add the single result as "stats"
        if len(experiment_runs) > 0 and experiment_runs[0].get("status") == "completed":
            single_run_stats = {
                "n_successful_runs": 1,
                "n_failed_runs": 0,
                "success_rate": 1.0,
                "experiment_name": experiment_runs[0]["experiment_name"],
                "experiment_description": experiment_runs[0]["experiment_description"],
            }
            
            # Copy single run values as "mean" values
            for key, value in experiment_runs[0].items():
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    single_run_stats[f"{key}_mean"] = value
                elif key in ["layer_mode", "model_class", "latent_dim", "num_layers", "channels_ratio"]:
                    single_run_stats[key] = value
            
            sensitivity_stats.append(single_run_stats)
    
    # Save individual experiment runs if requested
    if save_detailed_results:
        for run_idx, run_result in enumerate(experiment_runs):
            if enable_sensitivity_analysis and sensitivity_n_runs > 1:
                result_file = results_path / f"experiment_{i+1:02d}_{exp_config['name']}_run_{run_idx+1}.json"
            else:
                result_file = results_path / f"experiment_{i+1:02d}_{exp_config['name']}.json"
            
            with open(result_file, 'w') as f:
                # Convert numpy arrays to lists for JSON serialization
                json_result = copy.deepcopy(run_result)
                if 'training_history' in json_result and json_result['training_history']:
                    for key, values in json_result['training_history'].items():
                        if isinstance(values, np.ndarray):
                            json_result['training_history'][key] = values.tolist()
                json.dump(json_result, f, indent=2, default=str)
        
        # Save sensitivity statistics
        if enable_sensitivity_analysis and len(experiment_runs) > 1:
            stats_file = results_path / f"experiment_{i+1:02d}_{exp_config['name']}_sensitivity_stats.json"
            with open(stats_file, 'w') as f:
                json.dump(experiment_stats, f, indent=2, default=str)
            print(f"   💾 Sensitivity statistics saved to: {stats_file}")

# Calculate total time
total_time = time.time() - experiment_start_time
print(f"\n" + "="*80)
print(f"🎉 All experiments completed!")
print(f"   Total time: {total_time:.1f}s ({total_time/60:.1f} minutes)")

if enable_sensitivity_analysis:
    total_individual_runs = len(all_results)
    print(f"   Total individual runs: {total_individual_runs}")
    print(f"   Average time per run: {total_time/total_individual_runs:.1f}s")
else:
    print(f"   Average time per experiment: {total_time/len(experiment_configs):.1f}s")

# Count successful vs failed experiments and runs
successful_runs = sum(1 for r in all_results if r.get("status") == "completed")
failed_runs = len(all_results) - successful_runs
successful_experiments = sum(1 for s in sensitivity_stats if s.get("success_rate", 0) > 0)

print(f"   Successful experiments: {successful_experiments}/{len(experiment_configs)}")
print(f"   Successful runs: {successful_runs}/{len(all_results)}")

if failed_runs > 0:
    print(f"   Failed runs: {failed_runs}/{len(all_results)}")
    print("   Failed runs:")
    for r in all_results:
        if r.get("status") == "failed":
            exp_name = r.get('experiment_name', 'unknown')
            run_id = r.get('run_id', 0)
            error = r.get('error', 'unknown error')
            print(f"     - {exp_name} (run {run_id + 1}): {error}")

if enable_sensitivity_analysis:
    print(f"\n✅ Pipeline-based experiment execution with sensitivity analysis completed!")
    print(f"🔄 Each experiment was run {sensitivity_n_runs} times for statistical analysis")
else:
    print(f"\n✅ Pipeline-based experiment execution completed!")

print(f"🎯 All experiments used the standard ReconstructionExperiment pipeline")

## Results Aggregation and Analysis

Aggregate and analyze the results from all experiments to understand the impact of different pipeline configurations on reconstruction performance.

In [ ]:
# Create a comprehensive results DataFrame for analysis
def create_results_dataframe(results: List[Dict]) -> pd.DataFrame:
    """Create a pandas DataFrame from experiment results for analysis."""
    
    rows = []
    for result in results:
            
        # Extract basic info - use .get() to handle missing keys gracefully
        row = {
            "experiment_name": result.get("experiment_name", "unknown"),
            "description": result.get("experiment_description", "unknown"),
            "duration_seconds": result.get("duration_seconds", np.nan),
            "layer_mode": result.get("layer_mode", "unknown"),
            "model_class": result.get("model_class", "unknown"),
            "total_parameters": result.get("total_parameters", 0),
            "trainable_parameters": result.get("trainable_parameters", 0),
            "final_train_loss": result.get("final_train_loss", np.nan),
            "final_val_loss": result.get("final_val_loss", np.nan),
            "best_val_loss": result.get("best_val_loss", np.nan),
            "best_epoch": result.get("best_epoch", -1),
        }
        # Extract metrics with proper filtering
        if "metrics" in result:
            for metric_name, value in result["metrics"].items():
                if isinstance(value, dict):
                    # Flatten nested metrics
                    for sub_metric, sub_value in value.items():
                        if isinstance(sub_value, dict):
                            print(f"   Warning: Nested metric found for {metric_name}.{sub_metric}, skipping")
                        else:
                            # Skip complex metrics that shouldn't be in DataFrame
                            if any(substring in sub_metric for substring in ["confusion_matrix", "class_labels"]):
                                print(f"   Warning: Skipping complex metric {metric_name}.{sub_metric} for DataFrame")
                            else:
                                row[f"metric_{metric_name}_{sub_metric}"] = sub_value
                    # Also add the parent metric if it's not a dict
                    if not isinstance(value, dict):
                        row[f"metric_{metric_name}"] = value
                else:
                    row[f"metric_{metric_name}"] = value
        # Extract configuration details
        config = result.get("config", {})
        
        # Model config details
        model_config = config.get("model", {})
        row["num_layers"] = model_config.get("num_layers", np.nan)
        row["channels_ratio"] = model_config.get("channels_ratio", np.nan)
        row["latent_dim"] = model_config.get("latent_dim", np.nan)
        
        # Data config details  
        data_config = config.get("data", {})
        row["real_pipeline_type"] = data_config.get("real_pipeline_type", "none")
        row["num_transforms"] = len(data_config.get("transforms", []))
        
        # Training config details
        train_config = config.get("training", {})
        row["epochs"] = train_config.get("epochs", np.nan)
        row["learning_rate"] = train_config.get("learning_rate", np.nan)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

def create_sensitivity_dataframe(sensitivity_stats: List[Dict]) -> pd.DataFrame:
    """Create a pandas DataFrame from sensitivity analysis statistics."""
    
    if not sensitivity_stats:
        return pd.DataFrame()
    
    rows = []
    for stats in sensitivity_stats:
        # Extract basic experiment info - use .get() to handle missing keys
        row = {
            "experiment_name": stats.get("experiment_name", "unknown"),
            "description": stats.get("experiment_description", "unknown"),
            "n_successful_runs": stats.get("n_successful_runs", 0),
            "n_failed_runs": stats.get("n_failed_runs", 0),
            "success_rate": stats.get("success_rate", 0.0),
            "layer_mode": stats.get("layer_mode", "unknown"),
            "model_class": stats.get("model_class", "unknown"),
        }
        
        # Extract mean, std, and confidence intervals for key metrics
        key_metrics = [
            "final_train_loss", "final_val_loss", "best_val_loss", "best_epoch",
            "total_parameters", "trainable_parameters", "duration_seconds"
        ]
        
        for metric in key_metrics:
            row[f"{metric}_mean"] = stats.get(f"{metric}_mean", np.nan)
            row[f"{metric}_std"] = stats.get(f"{metric}_std", np.nan)
            row[f"{metric}_min"] = stats.get(f"{metric}_min", np.nan)
            row[f"{metric}_max"] = stats.get(f"{metric}_max", np.nan)
            row[f"{metric}_ci_lower"] = stats.get(f"{metric}_ci_lower", np.nan)
            row[f"{metric}_ci_upper"] = stats.get(f"{metric}_ci_upper", np.nan)
        
        # Extract evaluation metrics statistics
        for key, value in stats.items():
            if key.startswith("metric_") and key.endswith("_mean"):
                metric_base = key[:-5]  # Remove "_mean" suffix
                row[key] = value
                row[f"{metric_base}_std"] = stats.get(f"{metric_base}_std", np.nan)
                row[f"{metric_base}_ci_lower"] = stats.get(f"{metric_base}_ci_lower", np.nan)
                row[f"{metric_base}_ci_upper"] = stats.get(f"{metric_base}_ci_upper", np.nan)
        
        # Extract configuration details
        row["latent_dim"] = stats.get("latent_dim", np.nan)
        row["num_layers"] = stats.get("num_layers", np.nan)
        row["channels_ratio"] = stats.get("channels_ratio", np.nan)
        
        rows.append(row)
    
    return pd.DataFrame(rows)

# Create results DataFrames
if len(all_results) > 0:
    # Individual runs DataFrame (for detailed analysis)
    individual_results_df = create_results_dataframe(all_results)
    # Sensitivity statistics DataFrame (for statistical analysis)
    if enable_sensitivity_analysis and len(sensitivity_stats) > 0:
        results_df = create_sensitivity_dataframe(sensitivity_stats)
        print(f"📊 Sensitivity Analysis Results DataFrame created with {len(results_df)} experiments")
        print(f"   Each experiment represents statistics from {sensitivity_n_runs} runs")
        print(f"   Individual runs DataFrame: {len(individual_results_df)} total runs")
    else:
        results_df = individual_results_df
        print(f"📊 Results DataFrame created with {len(results_df)} experiments (single runs)")
    
    print(f"   Columns: {list(results_df.columns)}")
    
    # Save aggregated results
    if save_detailed_results:
        if enable_sensitivity_analysis and len(sensitivity_stats) > 0:
            # Save sensitivity statistics
            sensitivity_csv = results_path / "sensitivity_analysis_results.csv"
            results_df.to_csv(sensitivity_csv, index=False)
            print(f"   💾 Sensitivity analysis results saved to: {sensitivity_csv}")
            
            # Save individual runs
            individual_csv = results_path / "individual_runs_results.csv"
            individual_results_df.to_csv(individual_csv, index=False)
            print(f"   💾 Individual runs results saved to: {individual_csv}")
        else:
            results_csv = results_path / "aggregated_results.csv"
            results_df.to_csv(results_csv, index=False)
            print(f"   💾 Aggregated results saved to: {results_csv}")
    
    # Display basic statistics
    print(f"\n📈 Basic Statistics:")
    
    if enable_sensitivity_analysis and len(sensitivity_stats) > 0:
        print(f"\n   🔄 Sensitivity Analysis Summary:")
        print(f"     Experiments: {len(results_df)}")
        print(f"     Total runs: {len(individual_results_df)}")
        print(f"     Average success rate: {results_df['success_rate'].mean():.1%}")
        
        # Performance metrics summary with confidence intervals
        metric_cols = [col for col in results_df.columns if col.startswith("metric_") and col.endswith("_mean")]
        if metric_cols:
            print(f"\n   📊 Performance Metrics (with 95% Confidence Intervals):")
            for col in metric_cols:
                metric_name = col.replace("metric_", "").replace("_mean", "")
                std_col = col.replace("_mean", "_std")
                ci_lower_col = col.replace("_mean", "_ci_lower")
                ci_upper_col = col.replace("_mean", "_ci_upper")
                
                if results_df[col].notna().any():
                    mean_val = results_df[col].mean()
                    avg_std = results_df[std_col].mean() if std_col in results_df.columns else 0
                    
                    # Get average CI width if available
                    if ci_lower_col in results_df.columns and ci_upper_col in results_df.columns:
                        valid_ci = results_df[[ci_lower_col, ci_upper_col]].dropna()
                        if len(valid_ci) > 0:
                            avg_ci_lower = valid_ci[ci_lower_col].mean()
                            avg_ci_upper = valid_ci[ci_upper_col].mean()
                            print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f} (avg 95% CI: [{avg_ci_lower:.6f}, {avg_ci_upper:.6f}])")
                        else:
                            print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
                    else:
                        print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
        
        # Training metrics summary with confidence intervals
        print(f"\n   🎯 Training Metrics (with 95% Confidence Intervals):")
        train_cols = ["final_train_loss_mean", "final_val_loss_mean", "best_val_loss_mean"]
        for col in train_cols:
            if col in results_df.columns and results_df[col].notna().any():
                std_col = col.replace("_mean", "_std")
                ci_lower_col = col.replace("_mean", "_ci_lower")
                ci_upper_col = col.replace("_mean", "_ci_upper")
                
                mean_val = results_df[col].mean()
                avg_std = results_df[std_col].mean() if std_col in results_df.columns else 0
                
                metric_name = col.replace("_mean", "")
                if ci_lower_col in results_df.columns and ci_upper_col in results_df.columns:
                    valid_ci = results_df[[ci_lower_col, ci_upper_col]].dropna()
                    if len(valid_ci) > 0:
                        avg_ci_lower = valid_ci[ci_lower_col].mean()
                        avg_ci_upper = valid_ci[ci_upper_col].mean()
                        print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f} (avg 95% CI: [{avg_ci_lower:.6f}, {avg_ci_upper:.6f}])")
                    else:
                        print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
                else:
                    print(f"     {metric_name:20}: {mean_val:.6f} ± {avg_std:.6f}")
    else:
        # Standard analysis for single runs
        # Performance metrics summary
        metric_cols = [col for col in results_df.columns if col.startswith("metric_")]
        if metric_cols:
            print(f"\n   Performance Metrics:")
            for col in metric_cols:
                metric_name = col.replace("metric_", "")
                if results_df[col].notna().any():
                    mean_val = results_df[col].mean()
                    std_val = results_df[col].std()
                    min_val = results_df[col].min()
                    max_val = results_df[col].max()
                    print(f"     {metric_name:8}: {mean_val:.6f} ± {std_val:.6f} (range: {min_val:.6f} - {max_val:.6f})")
        
        # Training metrics summary
        print(f"\n   Training Metrics:")
        train_cols = ["final_train_loss", "final_val_loss", "best_val_loss"]
        for col in train_cols:
            if col in results_df.columns and results_df[col].notna().any():
                mean_val = results_df[col].mean()
                std_val = results_df[col].std()
                print(f"     {col:18}: {mean_val:.6f} ± {std_val:.6f}")
    
    # Model complexity summary
    print(f"\n   🏗️  Model Complexity:")
    if enable_sensitivity_analysis and "total_parameters_mean" in results_df.columns:
        mean_params = results_df["total_parameters_mean"].mean()
        std_params = results_df["total_parameters_std"].mean() if "total_parameters_std" in results_df.columns else 0
        print(f"     Total parameters:   {mean_params:,.0f} ± {std_params:,.0f}")
    elif "total_parameters" in results_df.columns:
        mean_params = results_df["total_parameters"].mean()
        std_params = results_df["total_parameters"].std()
        print(f"     Total parameters:   {mean_params:,.0f} ± {std_params:,.0f}")
    
    # Timing summary
    print(f"\n   ⏱️  Timing:")
    if enable_sensitivity_analysis and "duration_seconds_mean" in results_df.columns:
        mean_duration = results_df["duration_seconds_mean"].mean()
        std_duration = results_df["duration_seconds_std"].mean() if "duration_seconds_std" in results_df.columns else 0
        print(f"     Duration per experiment: {mean_duration:.1f} ± {std_duration:.1f} seconds")
    elif "duration_seconds" in results_df.columns:
        mean_duration = results_df["duration_seconds"].mean()
        std_duration = results_df["duration_seconds"].std()
        print(f"     Duration per experiment: {mean_duration:.1f} ± {std_duration:.1f} seconds")
    
    # Display first few rows
    print(f"\n📋 Sample Results (first 3 rows):")
    if enable_sensitivity_analysis:
        display_cols = ["experiment_name", "layer_mode", "model_class", "success_rate", "final_val_loss_mean"]
        metric_mean_cols = [col for col in results_df.columns if col.startswith("metric_") and col.endswith("_mean")][:2]
        display_cols.extend(metric_mean_cols)
    else:
        display_cols = ["experiment_name", "layer_mode", "model_class", "final_val_loss"]
        metric_cols = [col for col in results_df.columns if col.startswith("metric_")][:2]
        display_cols.extend(metric_cols)
    
    display_cols = [col for col in display_cols if col in results_df.columns]
    
    if len(results_df) > 0:
        display(results_df[display_cols].head(3))
    
    if enable_sensitivity_analysis:
        print(f"\n✅ Sensitivity analysis completed!")
        print(f"🔄 Each experiment provides mean ± std and 95% confidence intervals")
    else:
        print(f"\n✅ Results analysis completed!")
    
else:
    print("❌ No results to analyze - all experiments failed")

## Comprehensive Analysis Framework

This section provides dedicated analysis for each experiment group, focusing on comprehensive metrics comparison including:

- **Primary Metrics**: MSE, PSNR, SSIM for reconstruction quality
- **Classification Metrics**: H/Alpha and Cameron classification accuracies  
- **Efficiency Metrics**: Model parameters, training time, parameter efficiency
- **Convergence Analysis**: Training dynamics and best epoch analysis

Each analysis section follows a standardized structure for easy comparison and provides actionable insights.

## Sensitivity Analysis Visualization

### 🔄 Statistical Analysis Across Multiple Runs

This section provides dedicated visualization and analysis of the sensitivity analysis results. When experiments are run multiple times, we can assess:

- **Stability & Reproducibility**: How consistent are the results across different random seeds?
- **Confidence Intervals**: Statistical confidence in our measurements
- **Variability Patterns**: Which configurations are more or less stable?
- **Statistical Significance**: Are observed differences statistically meaningful?

The visualizations include error bars representing standard deviation and confidence intervals where applicable.

In [ ]:
# Sensitivity Analysis Visualization and Statistical Analysis
if enable_sensitivity_analysis and len(sensitivity_stats) > 0 and len(individual_results_df) > 0:
    print("🔄 Creating Sensitivity Analysis Visualizations...")
    
    # Create comprehensive sensitivity analysis visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    # 1. Success Rate Analysis
    ax = axes[0]
    if "success_rate" in results_df.columns:
        sns.barplot(data=results_df, x="experiment_name", y="success_rate", ax=ax, color='green')
        ax.set_title("Experiment Success Rate", fontweight='bold')
        ax.set_ylabel("Success Rate")
        ax.tick_params(axis='x', rotation=45)
        ax.set_ylim(0, 1.1)
        
        # Add percentage labels
        for i, v in enumerate(results_df["success_rate"]):
            ax.text(i, v + 0.02, f'{v:.1%}', ha='center', va='bottom')
    
    # 2. Validation Loss Variability
    ax = axes[1]
    if "final_val_loss_mean" in results_df.columns and "final_val_loss_std" in results_df.columns:
        # Create error bar plot
        x_pos = range(len(results_df))
        ax.errorbar(x_pos, results_df["final_val_loss_mean"], 
                   yerr=results_df["final_val_loss_std"], 
                   fmt='o', capsize=5, capthick=2, markersize=8)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(results_df["experiment_name"], rotation=45)
        ax.set_title("Validation Loss: Mean ± Std", fontweight='bold')
        ax.set_ylabel("Final Validation Loss")
        ax.grid(True, alpha=0.3)
    
    # 3. Coefficient of Variation (CV) Analysis
    ax = axes[2]
    if "final_val_loss_mean" in results_df.columns and "final_val_loss_std" in results_df.columns:
        # Calculate coefficient of variation (std/mean) - shows relative variability
        cv = results_df["final_val_loss_std"] / results_df["final_val_loss_mean"]
        results_df_with_cv = results_df.copy()
        results_df_with_cv['cv'] = cv
        
        sns.barplot(data=results_df_with_cv, x="experiment_name", y="cv", ax=ax, color='orange')
        ax.set_title("Coefficient of Variation (CV)", fontweight='bold')
        ax.set_ylabel("CV (std/mean)")
        ax.tick_params(axis='x', rotation=45)
        
        # Add CV values as labels
        for i, v in enumerate(cv):
            ax.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')
    
    # 4. Training Time Variability
    ax = axes[3]
    if "duration_seconds_mean" in results_df.columns and "duration_seconds_std" in results_df.columns:
        x_pos = range(len(results_df))
        ax.errorbar(x_pos, results_df["duration_seconds_mean"], 
                   yerr=results_df["duration_seconds_std"], 
                   fmt='s', capsize=5, capthick=2, markersize=8, color='purple')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(results_df["experiment_name"], rotation=45)
        ax.set_title("Training Time: Mean ± Std", fontweight='bold')
        ax.set_ylabel("Duration (seconds)")
        ax.grid(True, alpha=0.3)
    
    # 5. Performance Distribution Box Plot
    ax = axes[4]
    # Create box plot from individual runs
    validation_losses_by_exp = []
    experiment_names = []
    
    for exp_name in results_df["experiment_name"].unique():
        exp_runs = individual_results_df[individual_results_df["experiment_name"] == exp_name]
        if len(exp_runs) > 0 and "final_val_loss" in exp_runs.columns:
            losses = exp_runs["final_val_loss"].dropna().tolist()
            validation_losses_by_exp.extend(losses)
            experiment_names.extend([exp_name] * len(losses))
    
    if validation_losses_by_exp:
        box_data = pd.DataFrame({
            'experiment': experiment_names,
            'validation_loss': validation_losses_by_exp
        })
        sns.boxplot(data=box_data, x="experiment", y="validation_loss", ax=ax)
        ax.set_title("Validation Loss Distribution", fontweight='bold')
        ax.set_ylabel("Final Validation Loss")
        ax.tick_params(axis='x', rotation=45)
    
    # 6. Stability Ranking
    ax = axes[5]
    if "final_val_loss_std" in results_df.columns:
        # Sort by standard deviation (lower = more stable)
        stability_ranking = results_df.sort_values("final_val_loss_std")
        
        sns.barplot(data=stability_ranking, x="experiment_name", y="final_val_loss_std", ax=ax, color='red')
        ax.set_title("Stability Ranking (Lower = More Stable)", fontweight='bold')
        ax.set_ylabel("Standard Deviation")
        ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.suptitle("Sensitivity Analysis: Statistical Properties Across Multiple Runs", fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Statistical Analysis Summary
    print(f"\n📊 Statistical Analysis Summary:")
    print(f"   Total experiments: {len(results_df)}")
    print(f"   Runs per experiment: {sensitivity_n_runs}")
    print(f"   Total individual runs: {len(individual_results_df)}")
    
    # Overall success rate
    overall_success_rate = results_df["success_rate"].mean()
    print(f"   Overall success rate: {overall_success_rate:.1%}")
    
    # Stability analysis
    if "final_val_loss_std" in results_df.columns:
        most_stable_exp = results_df.loc[results_df["final_val_loss_std"].idxmin()]
        least_stable_exp = results_df.loc[results_df["final_val_loss_std"].idxmax()]
        
        print(f"\n🏆 Most Stable Configuration:")
        print(f"   Experiment: {most_stable_exp['experiment_name']}")
        print(f"   Std Dev: {most_stable_exp['final_val_loss_std']:.6f}")
        print(f"   Mean Loss: {most_stable_exp['final_val_loss_mean']:.6f}")
        
        print(f"\n⚠️  Least Stable Configuration:")
        print(f"   Experiment: {least_stable_exp['experiment_name']}")
        print(f"   Std Dev: {least_stable_exp['final_val_loss_std']:.6f}")
        print(f"   Mean Loss: {least_stable_exp['final_val_loss_mean']:.6f}")
        
        # Calculate coefficient of variation for all experiments
        cv_values = results_df["final_val_loss_std"] / results_df["final_val_loss_mean"]
        avg_cv = cv_values.mean()
        print(f"\n📈 Average Coefficient of Variation: {avg_cv:.3f}")
        print(f"   Interpretation: {'Low variability' if avg_cv < 0.1 else 'Moderate variability' if avg_cv < 0.2 else 'High variability'}")
    
    # Confidence interval analysis
    print(f"\n🎯 Confidence Interval Analysis (95% CI):")
    if "final_val_loss_ci_lower" in results_df.columns and "final_val_loss_ci_upper" in results_df.columns:
        for _, row in results_df.iterrows():
            exp_name = row["experiment_name"]
            mean_loss = row["final_val_loss_mean"]
            ci_lower = row["final_val_loss_ci_lower"]
            ci_upper = row["final_val_loss_ci_upper"]
            ci_width = ci_upper - ci_lower
            
            print(f"   {exp_name:25}: {mean_loss:.6f} [{ci_lower:.6f}, {ci_upper:.6f}] (width: {ci_width:.6f})")
    
    # Statistical significance tests (if multiple experiments)
    if len(results_df) >= 2:
        print(f"\n🧪 Statistical Significance Analysis:")
        
        # Perform pairwise t-tests between experiments
        from scipy.stats import ttest_ind
        
        exp_names = results_df["experiment_name"].tolist()
        significant_pairs = []
        
        for i in range(len(exp_names)):
            for j in range(i+1, len(exp_names)):
                exp1_name = exp_names[i]
                exp2_name = exp_names[j]
                
                # Get individual run results for both experiments
                exp1_runs = individual_results_df[individual_results_df["experiment_name"] == exp1_name]["final_val_loss"].dropna()
                exp2_runs = individual_results_df[individual_results_df["experiment_name"] == exp2_name]["final_val_loss"].dropna()
                
                if len(exp1_runs) >= 2 and len(exp2_runs) >= 2:
                    t_stat, p_value = ttest_ind(exp1_runs, exp2_runs)
                    
                    if p_value < 0.05:
                        significant_pairs.append((exp1_name, exp2_name, p_value))
                        print(f"   Significant difference between {exp1_name} and {exp2_name} (p={p_value:.4f})")
        
        if not significant_pairs:
            print(f"   No statistically significant differences found (α = 0.05)")
    
    print(f"\n✅ Sensitivity analysis visualization completed!")

elif enable_sensitivity_analysis:
    print("⚠️  Sensitivity analysis enabled but no results available")
else:
    print("ℹ️  Sensitivity analysis disabled - showing single run results only")

In [ ]:
# Helper functions for comprehensive metrics analysis
def create_comprehensive_analysis(filtered_data, title, experiment_var, x_label="Configuration"):
    """
    Create comprehensive analysis plots for a specific experiment group.
    Supports both single runs and sensitivity analysis with error bars.
    
    Args:
        filtered_data: DataFrame with experiment results for this group
        title: Title for the analysis section
        experiment_var: Variable being compared (e.g., 'layer_mode', 'latent_dim')
        x_label: Label for x-axis
    """
    if len(filtered_data) == 0:
        print(f"❌ No data available for {title}")
        return
    
    print(f"📊 Creating {title} with {len(filtered_data)} experiments")
    
    # Check if we have sensitivity analysis data
    has_sensitivity = any(col.endswith('_mean') for col in filtered_data.columns)
    
    # Set up comprehensive plotting grid - expanded to 4x4 for more metrics
    fig, axes = plt.subplots(4, 4, figsize=(24, 20))
    axes = axes.flatten()
    
    def plot_metric_with_error_bars(ax, data, x_var, y_var, color, title_text):
        """Helper function to plot metrics with error bars if available."""
        if has_sensitivity:
            # Use mean and std for sensitivity analysis
            mean_col = f"{y_var}_mean" if not y_var.endswith('_mean') else y_var
            std_col = f"{y_var}_std" if not y_var.endswith('_mean') else y_var.replace('_mean', '_std')
            
            if mean_col in data.columns:
                if experiment_var == 'latent_dim' and len(data) > 2:
                    # Line plot for latent dimension progression with error bars
                    sorted_data = data.sort_values(experiment_var)
                    if std_col in data.columns:
                        ax.errorbar(sorted_data[experiment_var], sorted_data[mean_col], 
                                  yerr=sorted_data[std_col], fmt='o-', linewidth=2, 
                                  markersize=8, color=color, capsize=5)
                    else:
                        ax.plot(sorted_data[experiment_var], sorted_data[mean_col], 
                               'o-', linewidth=2, markersize=8, color=color)
                    if experiment_var == 'latent_dim':
                        ax.set_xscale('log', base=2)
                else:
                    # Bar plot for categorical comparisons with error bars
                    if std_col in data.columns:
                        sns.barplot(data=data, x=experiment_var, y=mean_col, ax=ax, color=color)
                        # Add error bars manually
                        for i, (idx, row) in enumerate(data.iterrows()):
                            ax.errorbar(i, row[mean_col], yerr=row[std_col], 
                                      fmt='none', color='black', capsize=5)
                    else:
                        sns.barplot(data=data, x=experiment_var, y=mean_col, ax=ax, color=color)
                ax.set_ylabel(y_var.replace('_mean', '').replace('_', ' ').title())
        else:
            # Standard plotting for single runs
            if y_var in data.columns:
                if experiment_var == 'latent_dim' and len(data) > 2:
                    sorted_data = data.sort_values(experiment_var)
                    ax.plot(sorted_data[experiment_var], sorted_data[y_var], 'o-', linewidth=2, markersize=8, color=color)
                    if experiment_var == 'latent_dim':
                        ax.set_xscale('log', base=2)
                else:
                    sns.barplot(data=data, x=experiment_var, y=y_var, ax=ax, color=color)
                ax.set_ylabel(y_var.replace('_', ' ').title())
        
        ax.set_title(title_text, fontweight='bold')
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, alpha=0.3)
    
    # 1. Validation Loss Comparison
    plot_metric_with_error_bars(axes[0], filtered_data, experiment_var, "final_val_loss", 'red', "Validation Loss Comparison")
    
    # 2. MSE Metric (if available)
    mse_col = None
    for col in filtered_data.columns:
        if 'mse' in col.lower() and ('metric_' in col or col.startswith('metric_')):
            mse_col = col
            break
    
    if mse_col and filtered_data[mse_col].notna().any():
        plot_metric_with_error_bars(axes[1], filtered_data, experiment_var, mse_col, 'red', "MSE Comparison (Lower is Better)")
    else:
        axes[1].text(0.5, 0.5, 'MSE Data\nNot Available', ha='center', va='center', transform=axes[1].transAxes)
        axes[1].set_title("MSE Comparison", fontweight='bold')
    
    # 3. PSNR Metric (if available)
    psnr_col = None
    for col in filtered_data.columns:
        if 'psnr' in col.lower() and ('metric_' in col or col.startswith('metric_')):
            psnr_col = col
            break
    
    if psnr_col and filtered_data[psnr_col].notna().any():
        plot_metric_with_error_bars(axes[2], filtered_data, experiment_var, psnr_col, 'green', "PSNR Comparison (Higher is Better)")
    else:
        axes[2].text(0.5, 0.5, 'PSNR Data\nNot Available', ha='center', va='center', transform=axes[2].transAxes)
        axes[2].set_title("PSNR Comparison", fontweight='bold')
    
    # 4. SSIM Metric (if available)
    ssim_col = None
    for col in filtered_data.columns:
        if 'ssim' in col.lower() and ('metric_' in col or col.startswith('metric_')):
            ssim_col = col
            break
    
    if ssim_col and filtered_data[ssim_col].notna().any():
        plot_metric_with_error_bars(axes[3], filtered_data, experiment_var, ssim_col, 'blue', "SSIM Comparison (Higher is Better)")
    else:
        axes[3].text(0.5, 0.5, 'SSIM Data\nNot Available', ha='center', va='center', transform=axes[3].transAxes)
        axes[3].set_title("SSIM Comparison", fontweight='bold')
    
    # 5-8. Classification metrics (H/Alpha)
    h_alpha_metrics = [
        ('h_alpha_metrics_accuracy', 'H/Alpha Accuracy', 'purple'),
        ('h_alpha_metrics_f1_macro', 'H/Alpha F1 Macro', 'darkviolet'),
        ('h_alpha_metrics_cohen_kappa', 'H/Alpha Cohen\'s Kappa', 'indigo'),
        ('h_alpha_metrics_matthews_corrcoef', 'H/Alpha Matthews Corr.', 'mediumvioletred')
    ]
    
    for i, (metric_key, metric_title, color) in enumerate(h_alpha_metrics, 4):
        metric_col = None
        for col in filtered_data.columns:
            if metric_key in col.lower() and ('metric_' in col or col.startswith('metric_')):
                metric_col = col
                break
        
        if metric_col and filtered_data[metric_col].notna().any():
            plot_metric_with_error_bars(axes[i], filtered_data, experiment_var, metric_col, color, metric_title)
        else:
            axes[i].text(0.5, 0.5, f'{metric_title}\nNot Available', ha='center', va='center', transform=axes[i].transAxes)
            axes[i].set_title(metric_title, fontweight='bold')
    
    # 9-12. Cameron classification metrics
    cameron_metrics = [
        ('cameron_metrics_accuracy', 'Cameron Accuracy', 'orange'),
        ('cameron_metrics_f1_macro', 'Cameron F1 Macro', 'darkorange'),
        ('cameron_metrics_cohen_kappa', 'Cameron Cohen\'s Kappa', 'chocolate'),
        ('cameron_metrics_matthews_corrcoef', 'Cameron Matthews Corr.', 'sienna')
    ]
    
    for i, (metric_key, metric_title, color) in enumerate(cameron_metrics, 8):
        metric_col = None
        for col in filtered_data.columns:
            if metric_key in col.lower() and ('metric_' in col or col.startswith('metric_')):
                metric_col = col
                break
        
        if metric_col and filtered_data[metric_col].notna().any():
            plot_metric_with_error_bars(axes[i], filtered_data, experiment_var, metric_col, color, metric_title)
        else:
            axes[i].text(0.5, 0.5, f'{metric_title}\nNot Available', ha='center', va='center', transform=axes[i].transAxes)
            axes[i].set_title(metric_title, fontweight='bold')
    
    # 13. Model Parameters
    plot_metric_with_error_bars(axes[12], filtered_data, experiment_var, "total_parameters", 'brown', "Model Parameters")
    if "total_parameters" in filtered_data.columns or "total_parameters_mean" in filtered_data.columns:
        axes[12].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x/1000:.0f}K' if x < 1e6 else f'{x/1e6:.1f}M'))
    
    # 14. Training Time
    plot_metric_with_error_bars(axes[13], filtered_data, experiment_var, "duration_seconds", 'gray', "Training Time")
    
    # 15. Parameter Efficiency
    param_col = "total_parameters_mean" if has_sensitivity else "total_parameters"
    val_loss_col = "final_val_loss_mean" if has_sensitivity else "final_val_loss"
    
    if param_col in filtered_data.columns and val_loss_col in filtered_data.columns:
        # Calculate efficiency as inverse of (loss * parameters) - higher is better
        efficiency = 1 / (filtered_data[val_loss_col] * filtered_data[param_col] / 1e6)
        filtered_data_with_eff = filtered_data.copy()
        filtered_data_with_eff['efficiency'] = efficiency
        
        plot_metric_with_error_bars(axes[14], filtered_data_with_eff, experiment_var, "efficiency", 'darkred', "Parameter Efficiency")
        axes[14].set_ylabel("Efficiency (1/[Loss × M_params])")
    
    # Hide unused subplot
    axes[15].set_visible(False)
    
    plt.tight_layout()
    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.show()
    
    # Enhanced Summary table with all available metrics
    print(f"\n📊 {title} - Comprehensive Summary Table:")
    
    # Determine columns to display based on sensitivity analysis
    if has_sensitivity:
        summary_cols = ["experiment_name", "success_rate"]
        if "final_val_loss_mean" in filtered_data.columns:
            summary_cols.extend(["final_val_loss_mean", "final_val_loss_std"])
    else:
        summary_cols = ["experiment_name"]
        if "final_val_loss" in filtered_data.columns:
            summary_cols.append("final_val_loss")
    
    # Add available reconstruction metrics
    for metric_type in ['mse', 'psnr', 'ssim']:
        for col in filtered_data.columns:
            if metric_type in col.lower() and ('metric_' in col or col.startswith('metric_')):
                summary_cols.append(col)
                if has_sensitivity and col.endswith('_mean'):
                    std_col = col.replace('_mean', '_std')
                    if std_col in filtered_data.columns:
                        summary_cols.append(std_col)
                break
    
    # Add parameter info
    if has_sensitivity:
        if "total_parameters_mean" in filtered_data.columns:
            summary_cols.extend(["total_parameters_mean", "total_parameters_std"])
    else:
        if "total_parameters" in filtered_data.columns:
            summary_cols.append("total_parameters")
    
    # Display available columns only
    available_cols = [col for col in summary_cols if col in filtered_data.columns]
    
    if len(available_cols) > 1:
        display_data = filtered_data[available_cols].round(6)
        display(display_data)
    
    # Print sensitivity analysis insights
    if has_sensitivity and len(filtered_data) > 1:
        print(f"\n🔄 Sensitivity Analysis Insights:")
        
        # Stability analysis
        val_loss_std_col = "final_val_loss_std"
        if val_loss_std_col in filtered_data.columns:
            most_stable_idx = filtered_data[val_loss_std_col].idxmin()
            least_stable_idx = filtered_data[val_loss_std_col].idxmax()
            
            most_stable = filtered_data.loc[most_stable_idx]
            least_stable = filtered_data.loc[least_stable_idx]
            
            print(f"   📈 Most Stable: {most_stable['experiment_name']} (std: {most_stable[val_loss_std_col]:.6f})")
            print(f"   📉 Least Stable: {least_stable['experiment_name']} (std: {least_stable[val_loss_std_col]:.6f})")
            
            avg_std = filtered_data[val_loss_std_col].mean()
            print(f"   📊 Average Variability: {avg_std:.6f}")
    
    return filtered_data

print("✅ Enhanced comprehensive analysis helper functions defined with sensitivity analysis support")
print("🔄 Includes error bars, confidence intervals, and stability analysis")

## 1. Layer Mode Comparison (AutoEncoder)

### 🔄 Complex vs Real Processing for AutoEncoder Architecture

This analysis compares how different layer modes (complex vs real) affect AutoEncoder performance. We evaluate:

**Key Comparisons:**
- **Complex Native**: Direct complex number processing
- **Real Dual Channel**: Real/imaginary channels processed separately

**Metrics Analyzed:**
- **Reconstruction Quality**: MSE, PSNR, SSIM
- **Classification Performance**: H/Alpha and Cameron decomposition accuracy
- **Computational Efficiency**: Model parameters and training time

In [ ]:
# 1. Layer Mode Comparison (AutoEncoder) Analysis
if len(all_results) > 0 and len(results_df) > 0:
    # Filter for AutoEncoder layer mode experiments
    if 'layer_mode_experiments_ae' in locals():
        ae_layer_names = [exp['name'] for exp in layer_mode_experiments_ae]
        ae_layer_data = results_df[
            results_df["experiment_name"].isin(ae_layer_names)
        ].copy()
        
        if len(ae_layer_data) > 0:
            # Create comprehensive analysis
            ae_layer_data = create_comprehensive_analysis(
                ae_layer_data, 
                "Layer Mode Comparison (AutoEncoder)", 
                "layer_mode",
                "Layer Mode"
            )
            
            # Specific insights for layer mode comparison
            print(f"\n🔍 Layer Mode (AutoEncoder) Insights:")
            
            # Determine which columns to use based on sensitivity analysis
            val_loss_col = "final_val_loss_mean" if enable_sensitivity_analysis else "final_val_loss"
            param_col = "total_parameters_mean" if enable_sensitivity_analysis else "total_parameters"
            
            if val_loss_col in ae_layer_data.columns:
                best_mode_idx = ae_layer_data[val_loss_col].idxmin()
                best_mode_exp = ae_layer_data.loc[best_mode_idx]
                
                print(f"   🏆 Best Layer Mode: {best_mode_exp['layer_mode']}")
                if enable_sensitivity_analysis:
                    val_loss_mean = best_mode_exp[val_loss_col]
                    val_loss_std = best_mode_exp.get("final_val_loss_std", 0)
                    print(f"   📊 Validation Loss: {val_loss_mean:.6f} ± {val_loss_std:.6f}")
                    if "success_rate" in best_mode_exp:
                        print(f"   ✅ Success Rate: {best_mode_exp['success_rate']:.1%}")
                else:
                    print(f"   📊 Validation Loss: {best_mode_exp[val_loss_col]:.6f}")
                
                # Compare modes
                mode_comparison = ae_layer_data.groupby("layer_mode")[val_loss_col].agg(['mean', 'std', 'min'])
                print(f"\n   📈 Performance by Layer Mode:")
                for mode, stats in mode_comparison.iterrows():
                    if enable_sensitivity_analysis:
                        print(f"     {mode:10}: Loss = {stats['mean']:.6f} ± {stats['std']:.6f}")
                    else:
                        print(f"     {mode:10}: Loss = {stats['mean']:.6f} ± {stats['std']:.6f}")
                
                # Parameter comparison
                if param_col in ae_layer_data.columns:
                    param_comparison = ae_layer_data.groupby("layer_mode")[param_col].mean()
                    print(f"\n   🏗️  Parameters by Layer Mode:")
                    for mode, params in param_comparison.items():
                        print(f"     {mode:10}: {params:,.0f} parameters")
                
                # Sensitivity analysis specific insights
                if enable_sensitivity_analysis and "final_val_loss_std" in ae_layer_data.columns:
                    print(f"\n   🔄 Stability Analysis:")
                    stability_comparison = ae_layer_data.groupby("layer_mode")["final_val_loss_std"].mean()
                    for mode, std_dev in stability_comparison.items():
                        print(f"     {mode:10}: Std Dev = {std_dev:.6f}")
                    
                    most_stable_mode = stability_comparison.idxmin()
                    print(f"     Most stable: {most_stable_mode}")
            
            print(f"\n   💡 Recommendations:")
            if len(ae_layer_data) >= 2:
                complex_data = ae_layer_data[ae_layer_data["layer_mode"] == "complex"]
                real_data = ae_layer_data[ae_layer_data["layer_mode"] == "real"]
                
                if len(complex_data) > 0 and len(real_data) > 0:
                    complex_loss = complex_data[val_loss_col].iloc[0]
                    real_loss = real_data[val_loss_col].iloc[0]
                    
                    if enable_sensitivity_analysis:
                        # Check if difference is statistically significant
                        complex_std = complex_data.get("final_val_loss_std", pd.Series([0])).iloc[0]
                        real_std = real_data.get("final_val_loss_std", pd.Series([0])).iloc[0]
                        
                        # Simple significance check using standard errors
                        difference = abs(complex_loss - real_loss)
                        combined_std = np.sqrt(complex_std**2 + real_std**2)
                        
                        if difference > 2 * combined_std:  # Rough 95% confidence
                            significance_note = " (statistically significant)"
                        else:
                            significance_note = " (not statistically significant)"
                        
                        print(f"     - Loss difference: {difference:.6f}{significance_note}")
                    
                    if complex_loss < real_loss:
                        print(f"     - Complex processing shows better reconstruction quality")
                        print(f"     - Consider complex layers for optimal SAR reconstruction")
                    else:
                        print(f"     - Real processing achieves competitive performance")
                        print(f"     - Real layers may be preferred for computational efficiency")
                        
                    if enable_sensitivity_analysis:
                        print(f"     - Refer to stability analysis for reproducibility considerations")
        else:
            print("❌ No AutoEncoder layer mode experiments found in results")
    else:
        print("❌ AutoEncoder layer mode experiments not defined")
else:
    print("❌ No results available for analysis")

## 2. Layer Mode Comparison (LatentAutoEncoder)

### 🔄 Complex vs Real Processing for LatentAutoEncoder Architecture

This analysis compares how different layer modes affect LatentAutoEncoder performance with latent space compression. We evaluate:

**Key Comparisons:**
- **Complex Native**: Direct complex processing with latent compression
- **Real Dual Channel**: Real/imaginary processing with latent compression

**Focus Areas:**
- **Compression Impact**: How layer mode affects latent space effectiveness
- **Quality vs Compression**: Trade-offs in different processing modes
- **Latent Representation**: Quality of compressed representations by mode

In [ ]:
# 2. Layer Mode Comparison (LatentAutoEncoder) Analysis
if len(all_results) > 0 and len(results_df) > 0:
    
    # Filter for LatentAutoEncoder layer mode experiments
    if 'layer_mode_experiments_latent_ae' in locals():
        latent_ae_layer_names = [exp['name'] for exp in layer_mode_experiments_latent_ae]
        latent_ae_layer_data = results_df[
            results_df["experiment_name"].isin(latent_ae_layer_names)
        ].copy()
        
        if len(latent_ae_layer_data) > 0:
            # Create comprehensive analysis
            latent_ae_layer_data = create_comprehensive_analysis(
                latent_ae_layer_data, 
                "Layer Mode Comparison (LatentAutoEncoder)", 
                "layer_mode",
                "Layer Mode"
            )
            
            # Specific insights for LatentAutoEncoder layer mode comparison
            print(f"\n🔍 Layer Mode (LatentAutoEncoder) Insights:")
            
            # Determine which columns to use based on sensitivity analysis
            val_loss_col = "final_val_loss_mean" if enable_sensitivity_analysis else "final_val_loss"
            
            if val_loss_col in latent_ae_layer_data.columns:
                best_mode_idx = latent_ae_layer_data[val_loss_col].idxmin()
                best_mode_exp = latent_ae_layer_data.loc[best_mode_idx]
                
                print(f"   🏆 Best Layer Mode: {best_mode_exp['layer_mode']}")
                if enable_sensitivity_analysis:
                    val_loss_mean = best_mode_exp[val_loss_col]
                    val_loss_std = best_mode_exp.get("final_val_loss_std", 0)
                    print(f"   📊 Validation Loss: {val_loss_mean:.6f} ± {val_loss_std:.6f}")
                    if "success_rate" in best_mode_exp:
                        print(f"   ✅ Success Rate: {best_mode_exp['success_rate']:.1%}")
                else:
                    print(f"   📊 Validation Loss: {best_mode_exp[val_loss_col]:.6f}")
                print(f"   🔬 Latent Dimension: {int(best_mode_exp.get('latent_dim', 128))}D")
                
                # Compare modes for LatentAutoEncoder
                mode_comparison = latent_ae_layer_data.groupby("layer_mode")[val_loss_col].agg(['mean', 'std', 'min'])
                print(f"\n   📈 Performance by Layer Mode (with Latent Compression):")
                for mode, stats in mode_comparison.iterrows():
                    print(f"     {mode:10}: Loss = {stats['mean']:.6f} ± {stats['std']:.6f}")
                
                # Compression efficiency comparison
                print(f"\n   🗜️  Compression Efficiency Analysis:")
                for _, row in latent_ae_layer_data.iterrows():
                    mode = row['layer_mode']
                    loss = row[val_loss_col]
                    latent_d = int(row.get('latent_dim', 128))
                    
                    # Estimate compression ratio
                    estimated_input_size = 32 * 32 * 3  # Approximate
                    compression_ratio = estimated_input_size / latent_d
                    
                    if enable_sensitivity_analysis and "final_val_loss_std" in row:
                        loss_std = row["final_val_loss_std"]
                        print(f"     {mode:10}: {compression_ratio:.1f}x compression, Loss = {loss:.6f} ± {loss_std:.6f}")
                    else:
                        print(f"     {mode:10}: {compression_ratio:.1f}x compression, Loss = {loss:.6f}")
                
                # Sensitivity analysis for latent compression
                if enable_sensitivity_analysis and "final_val_loss_std" in latent_ae_layer_data.columns:
                    print(f"\n   🔄 Compression Stability Analysis:")
                    for _, row in latent_ae_layer_data.iterrows():
                        mode = row['layer_mode']
                        std_dev = row["final_val_loss_std"]
                        cv = std_dev / row[val_loss_col]  # Coefficient of variation
                        print(f"     {mode:10}: CV = {cv:.3f} ({'stable' if cv < 0.1 else 'moderate' if cv < 0.2 else 'variable'})")
            
            print(f"\n   💡 LatentAutoEncoder Layer Mode Recommendations:")
            if len(latent_ae_layer_data) >= 2:
                complex_data = latent_ae_layer_data[latent_ae_layer_data["layer_mode"] == "complex"]
                real_data = latent_ae_layer_data[latent_ae_layer_data["layer_mode"] == "real"]
                
                if len(complex_data) > 0 and len(real_data) > 0:
                    complex_loss = complex_data[val_loss_col].iloc[0]
                    real_loss = real_data[val_loss_col].iloc[0]
                    
                    # Check statistical significance if sensitivity analysis is enabled
                    if enable_sensitivity_analysis and "final_val_loss_std" in complex_data.columns:
                        complex_std = complex_data["final_val_loss_std"].iloc[0]
                        real_std = real_data["final_val_loss_std"].iloc[0]
                        difference = abs(complex_loss - real_loss)
                        combined_std = np.sqrt(complex_std**2 + real_std**2)
                        
                        is_significant = difference > 2 * combined_std
                        
                        print(f"     - Performance difference: {difference:.6f} {'(significant)' if is_significant else '(not significant)'}")
                        
                        if not is_significant:
                            print(f"     - Both modes achieve statistically similar compression quality")
                            print(f"     - Choice can be based on computational preferences and stability")
                        elif complex_loss < real_loss:
                            print(f"     - Complex processing significantly better preserves information during compression")
                            print(f"     - Recommended for applications requiring high reconstruction fidelity")
                        else:
                            print(f"     - Real processing shows significantly better latent space utilization")
                            print(f"     - May be preferred for interpretable latent representations")
                    else:
                        # Simple comparison without statistical testing
                        if abs(complex_loss - real_loss) < 0.01:
                            print(f"     - Both modes achieve similar compression quality")
                            print(f"     - Choice can be based on computational preferences")
                        elif complex_loss < real_loss:
                            print(f"     - Complex processing better preserves information during compression")
                            print(f"     - Recommended for applications requiring high reconstruction fidelity")
                        else:
                            print(f"     - Real processing shows better latent space utilization")
                            print(f"     - May be preferred for interpretable latent representations")
                    
                    if enable_sensitivity_analysis:
                        print(f"     - Consider stability metrics when choosing between similar-performing modes")
        else:
            print("❌ No LatentAutoEncoder layer mode experiments found in results")
    else:
        print("❌ LatentAutoEncoder layer mode experiments not defined")
else:
    print("❌ No results available for analysis")

## 3. Latent Vector Size Impact Analysis

### 🔬 Studying Compression vs Quality Trade-offs

This analysis examines how latent vector dimensionality affects LatentAutoEncoder performance across all key metrics. We systematically evaluate:

**Key Research Questions:**
- **Compression vs Quality**: How does reducing latent dimensionality impact reconstruction quality?
- **Parameter Efficiency**: What's the relationship between latent size and total model parameters?
- **Classification Preservation**: How well do different latent sizes preserve classification accuracy?
- **Optimal Balance**: What latent size provides the best trade-off between compression and quality?

**Latent Dimensions Tested:**
- **16D**: Maximum compression (tiny latent space)
- **32D**: High compression (small latent space)  
- **64D**: Moderate compression (medium latent space) - baseline
- **128D**: Low compression (large latent space)
- **256D**: Minimal compression (extra large latent space)
- **512D**: Almost no compression (huge latent space)

**Comprehensive Metrics:**
- **Reconstruction**: MSE, PSNR, SSIM for image quality assessment
- **Classification**: H/Alpha and Cameron decomposition accuracy preservation
- **Efficiency**: Parameter count, training time, and parameter efficiency analysis

In [ ]:
# 3. Latent Vector Size Impact Analysis
if len(all_results) > 0 and len(results_df) > 0:
    
    # Filter for latent size experiments
    if 'latent_size_experiments' in locals():
        latent_experiment_names = [exp['name'] for exp in latent_size_experiments]
        latent_experiments = results_df[
            results_df["experiment_name"].isin(latent_experiment_names)
        ].copy()
        
        if len(latent_experiments) > 0:
            print(f"🔬 Found {len(latent_experiments)} dedicated LatentAutoEncoder size experiments")
            
            # Sort by latent dimension for progression analysis
            latent_experiments = latent_experiments.sort_values("latent_dim")
            
            # Create comprehensive analysis using the new framework
            latent_experiments = create_comprehensive_analysis(
                latent_experiments, 
                "Latent Vector Size Impact Analysis", 
                "latent_dim",
                "Latent Dimension"
            )
            
            # Enhanced latent size specific analysis
            print(f"\n🔍 Detailed Latent Size Analysis:")
            
            # Find optimal latent dimension
            if "final_val_loss" in latent_experiments.columns:
                best_latent_idx = latent_experiments["final_val_loss"].idxmin()
                best_latent_exp = latent_experiments.loc[best_latent_idx]
                
                print(f"   🏆 Optimal Latent Dimension: {int(best_latent_exp['latent_dim'])}D")
                print(f"   📊 Best Validation Loss: {best_latent_exp['final_val_loss']:.6f}")
                print(f"   🏗️  Parameters: {best_latent_exp.get('total_parameters', 0):,}")
            
            # Compression analysis across all dimensions
            print(f"\n   📊 Compression Analysis Across Latent Sizes:")
            print(f"   {'Latent':>8} {'Loss':>10} {'PSNR':>8} {'SSIM':>8} {'Params':>10} {'Compression':>12}")
            print("-" * 70)
            
            for _, row in latent_experiments.iterrows():
                latent_dim = int(row["latent_dim"])
                val_loss = row.get("final_val_loss", float('nan'))
                
                # Find PSNR
                psnr_val = float('nan')
                for col in row.index:
                    if 'psnr' in col.lower() and col.startswith('metric_') and not pd.isna(row[col]):
                        psnr_val = row[col]
                        break
                
                # Find SSIM
                ssim_val = float('nan')
                for col in row.index:
                    if 'ssim' in col.lower() and col.startswith('metric_') and not pd.isna(row[col]):
                        ssim_val = row[col]
                        break
                
                params = row.get("total_parameters", 0)
                
                # Calculate compression ratio
                estimated_input_size = 32 * 32 * 3
                compression_ratio = estimated_input_size / latent_dim
                
                print(f"   {latent_dim:>8d} {val_loss:>10.6f} {psnr_val:>8.2f} {ssim_val:>8.4f} {params:>10,d} {compression_ratio:>12.1f}x")
            
            # Quality vs Compression Trade-off Analysis
            print(f"\n   🔍 Quality vs Compression Trade-off:")
            smallest_latent = latent_experiments.loc[latent_experiments["latent_dim"].idxmin()]
            largest_latent = latent_experiments.loc[latent_experiments["latent_dim"].idxmax()]
            
            if "final_val_loss" in latent_experiments.columns:
                loss_diff = largest_latent["final_val_loss"] - smallest_latent["final_val_loss"]
                param_ratio = largest_latent["total_parameters"] / smallest_latent["total_parameters"]
                
                print(f"     Extreme Compression ({int(smallest_latent['latent_dim'])}D): Loss = {smallest_latent['final_val_loss']:.6f}")
                print(f"     Minimal Compression ({int(largest_latent['latent_dim'])}D): Loss = {largest_latent['final_val_loss']:.6f}")
                print(f"     Quality Improvement: {-loss_diff:.6f} ({-loss_diff/largest_latent['final_val_loss']*100:.1f}%)")
                print(f"     Parameter Cost: {param_ratio:.1f}x increase")
            
            # Classification performance analysis
            print(f"\n   🎯 Classification Performance Analysis:")
            
            # Check for classification metrics
            h_alpha_available = any('h_alpha' in col.lower() or 'halpha' in col.lower() 
                                  for col in latent_experiments.columns if col.startswith('metric_'))
            cameron_available = any('cameron' in col.lower() 
                                  for col in latent_experiments.columns if col.startswith('metric_'))
            
            if h_alpha_available or cameron_available:
                print(f"   {'Latent':>8} {'H/Alpha':>10} {'Cameron':>10}")
                print("-" * 30)
                
                for _, row in latent_experiments.iterrows():
                    latent_dim = int(row["latent_dim"])
                    
                    # Find H/Alpha accuracy
                    h_alpha_val = float('nan')
                    for col in row.index:
                        if ('h_alpha' in col.lower() or 'halpha' in col.lower()) and col.startswith('metric_'):
                            if not pd.isna(row[col]):
                                h_alpha_val = row[col]
                                break
                    
                    # Find Cameron accuracy
                    cameron_val = float('nan')
                    for col in row.index:
                        if 'cameron' in col.lower() and col.startswith('metric_'):
                            if not pd.isna(row[col]):
                                cameron_val = row[col]
                                break
                    
                    print(f"   {latent_dim:>8d} {h_alpha_val:>10.4f} {cameron_val:>10.4f}")
            else:
                print("     Classification metrics not available in current results")
                        
        else:
            print("❌ No dedicated latent size experiments found in results")
    else:
        print("❌ Latent size experiments not defined")
else:
    print("❌ No results available for analysis")

print(f"\n✅ Comprehensive latent vector size analysis completed!")

In [ ]:
transform_names = [exp['name'] for exp in transform_experiments]
transform_data = results_df[
    results_df["experiment_name"].isin(transform_names)
].copy()
transform_data

## 4. Data Transform Configuration Analysis

### 🔄 Preprocessing and Augmentation Impact Study

This analysis evaluates how different data preprocessing and augmentation strategies affect model performance. We examine:

**Key Comparisons:**
- **Basic Pipeline**: Minimal preprocessing (dataset defaults only)
- **Enhanced Pipeline**: Additional transforms like resizing, normalization
- **Augmentation Pipeline**: Data augmentation for robustness

**Impact Areas:**
- **Data Quality**: How preprocessing affects input data characteristics
- **Model Robustness**: Impact of augmentation on generalization
- **Training Efficiency**: Processing overhead vs performance gains
- **Reconstruction Fidelity**: Preservation of SAR-specific features

In [ ]:
# 4. Data Transform Configuration Analysis
if len(all_results) > 0 and len(results_df) > 0:
    
    # Filter for transform experiments
    if 'transform_experiments' in locals():
        transform_names = [exp['name'] for exp in transform_experiments]
        transform_data = results_df[
            results_df["experiment_name"].isin(transform_names)
        ].copy()
        
        if len(transform_data) > 0:
            # Create num_transforms column based on experiment names (more robust approach)
            def count_transforms_from_name(name):
                """Count transforms based on experiment name patterns."""
                name_lower = name.lower()
                if 'basic' in name_lower:
                    return 0
                elif 'with' in name_lower:  # e.g., "WithResize", "WithRandomHorizontalFlip"
                    # Count number of transform words after "with"
                    # This is more generic than hardcoding specific transform names
                    return 1
                elif 'augment' in name_lower:
                    return 2
                else:
                    # Default: if it's not basic, assume it has at least 1 transform
                    return 1
                    
            transform_data['num_transforms'] = transform_data['experiment_name'].apply(count_transforms_from_name)
            
            # Debug: Show what num_transforms was assigned
            print(f"🔍 Transform assignment:")
            for _, row in transform_data.iterrows():
                print(f"   {row['experiment_name']} → {row['num_transforms']} transforms")
            
            # Create comprehensive analysis
            transform_data = create_comprehensive_analysis(
                transform_data, 
                "Data Transform Configuration Analysis", 
                "num_transforms",
                "Number of Transforms"
            )
            
            # Specific insights for transform configuration
            print(f"\n🔍 Transform Configuration Insights:")
            
            if "final_val_loss_mean" in transform_data.columns:
                best_transform_idx = transform_data["final_val_loss_mean"].idxmin()
                best_transform_exp = transform_data.loc[best_transform_idx]
                
                print(f"   🏆 Best Transform Config: {best_transform_exp['experiment_name']}")
                print(f"   📊 Validation Loss: {best_transform_exp['final_val_loss_mean']:.6f}")
                print(f"   🔄 Number of Transforms: {int(best_transform_exp.get('num_transforms', 0))}")
                
                # Compare transform configurations
                transform_comparison = transform_data.groupby("num_transforms")["final_val_loss_mean"].agg(['mean', 'std', 'min'])
                print(f"\n   📈 Performance by Transform Configuration:")
                for num_transforms, stats in transform_comparison.iterrows():
                    config_type = "Basic" if num_transforms == 0 else f"Enhanced ({num_transforms} transforms)"
                    print(f"     {config_type:20}: Loss = {stats['mean']:.6f} ± {stats['std']:.6f}")
                
                # Training time impact
                if "duration_seconds_mean" in transform_data.columns:
                    time_comparison = transform_data.groupby("num_transforms")["duration_seconds_mean"].mean()
                    print(f"\n   ⏱️  Training Time Impact:")
                    for num_transforms, time_val in time_comparison.items():
                        config_type = "Basic" if num_transforms == 0 else f"Enhanced ({num_transforms} transforms)"
                        print(f"     {config_type:20}: {time_val:.1f} seconds")
            
            print(f"\n   💡 Transform Configuration Recommendations:")
            
            if len(transform_data) >= 2:
                # Compare basic vs enhanced
                basic_data = transform_data[transform_data["num_transforms"] == 0]
                enhanced_data = transform_data[transform_data["num_transforms"] > 0]
                
                if len(basic_data) > 0 and len(enhanced_data) > 0:
                    basic_loss = basic_data["final_val_loss_mean"].iloc[0]
                    enhanced_loss = enhanced_data["final_val_loss_mean"].iloc[0]
                    
                    basic_time = basic_data.get("duration_seconds_mean", pd.Series([0])).iloc[0]
                    enhanced_time = enhanced_data.get("duration_seconds_mean", pd.Series([0])).iloc[0]
                    
                    loss_improvement = (basic_loss - enhanced_loss) / basic_loss * 100
                    time_overhead = (enhanced_time - basic_time) / basic_time * 100 if basic_time > 0 else 0
                    
                    print(f"     📊 Enhanced transforms vs Basic:")
                    print(f"       Quality improvement: {loss_improvement:.1f}%")
                    print(f"       Time overhead: {time_overhead:.1f}%")
                    
                    if loss_improvement > 5:  # Significant improvement
                        print(f"     ✅ Recommendation: Use enhanced transform pipeline")
                        print(f"       - Significant quality improvement justifies overhead")
                    elif loss_improvement > 0:
                        print(f"     ⚖️  Recommendation: Consider use case requirements")
                        print(f"       - Modest improvement, evaluate time vs quality trade-off")
                    else:
                        print(f"     🚫 Recommendation: Stick with basic transforms")
                        print(f"       - Enhanced transforms don't provide benefit for this data")
                    
                    # SAR-specific considerations
                    print(f"\n     📡 SAR-Specific Considerations:")
                    print(f"       - Geometric transforms may affect polarimetric relationships")
                    print(f"       - Intensity-based transforms should preserve SAR characteristics")
                    print(f"       - Consider domain-specific augmentations for robustness")
        else:
            print("❌ No transform configuration experiments found in results")
    else:
        print("❌ Transform experiments not defined (evaluate_transforms=False)")
        print("   Set evaluate_transforms=True to run transform comparison experiments")
else:
    print("❌ No results available for analysis")

## 5. Hyperparameter Impact Analysis

### ⚙️ Model Architecture and Training Configuration Study

This analysis examines how different hyperparameter settings affect model performance and training dynamics. We evaluate:

**Key Hyperparameters:**
- **Model Depth**: Number of layers (network capacity)
- **Channel Ratio**: Feature map scaling (model width)
- **Learning Rate**: Optimization step size
- **Training Duration**: Epoch count and convergence

**Analysis Focus:**
- **Capacity vs Overfitting**: Finding optimal model complexity
- **Training Efficiency**: Convergence speed and stability
- **Parameter Efficiency**: Performance per parameter analysis
- **Generalization**: How hyperparameters affect test performance

In [ ]:
# 5. Hyperparameter Impact Analysis
if len(all_results) > 0 and len(results_df) > 0:
    
    # Filter for hyperparameter experiments
    if 'hyperparam_experiments' in locals():
        hyperparam_names = [exp['name'] for exp in hyperparam_experiments]
        hyperparam_data = results_df[
            results_df["experiment_name"].isin(hyperparam_names)
        ].copy()
        
        if len(hyperparam_data) > 0:
            # Create comprehensive analysis
            hyperparam_data = create_comprehensive_analysis(
                hyperparam_data, 
                "Hyperparameter Impact Analysis", 
                "num_layers",
                "Model Configuration"
            )
            
            # Specific insights for hyperparameter tuning
            print(f"\n🔍 Hyperparameter Configuration Insights:")
            
            if "final_val_loss" in hyperparam_data.columns:
                best_hyperparam_idx = hyperparam_data["final_val_loss"].idxmin()
                best_hyperparam_exp = hyperparam_data.loc[best_hyperparam_idx]
                
                print(f"   🏆 Best Hyperparameter Config: {best_hyperparam_exp['experiment_name']}")
                print(f"   📊 Validation Loss: {best_hyperparam_exp['final_val_loss']:.6f}")
                print(f"   🏗️  Layers: {int(best_hyperparam_exp.get('num_layers', 3))}")
                print(f"   📏 Channels Ratio: {int(best_hyperparam_exp.get('channels_ratio', 32))}")
                print(f"   🧮 Total Parameters: {best_hyperparam_exp.get('total_parameters', 0):,}")
                
                # Model complexity analysis
                print(f"\n   📊 Model Complexity Analysis:")
                for _, row in hyperparam_data.iterrows():
                    exp_name = row['experiment_name']
                    layers = int(row.get('num_layers', 3))
                    channels = int(row.get('channels_ratio', 32))
                    params = row.get('total_parameters', 0)
                    val_loss = row.get('final_val_loss', float('nan'))
                    
                    complexity_score = layers * channels
                    efficiency = 1 / (val_loss * params / 1e6) if not np.isnan(val_loss) and params > 0 else 0
                    
                    print(f"     {exp_name:20}: {layers}L×{channels}C, {params:,} params, Loss={val_loss:.6f}, Eff={efficiency:.3f}")
                
                # Training dynamics analysis
                if "best_epoch" in hyperparam_data.columns:
                    print(f"\n   🎯 Training Dynamics:")
                    convergence_analysis = hyperparam_data.groupby("num_layers")["best_epoch"].agg(['mean', 'std'])
                    for layers, stats in convergence_analysis.iterrows():
                        print(f"     {int(layers)} layers: Converges at epoch {stats['mean']:.1f} ± {stats['std']:.1f}")
                            
            print(f"\n   💡 Hyperparameter Recommendations:")                    
                    
        else:
            print("❌ No hyperparameter experiments found in results")
    else:
        print("❌ Hyperparameter experiments not defined (evaluate_hyperparams=False)")
        print("   Set evaluate_hyperparams=True to run hyperparameter comparison experiments")
else:
    print("❌ No results available for analysis")

print(f"\n✅ All five comprehensive analysis sections completed!")
print(f"🎯 Analysis framework provides detailed insights for each experiment group")

In [ ]:
# Save final experiment summary
if len(all_results) > 0:
    summary = {
        "notebook_execution": {
            "timestamp": datetime.now().isoformat(),
            "total_experiments": len(experiment_configs),
            "successful_experiments": successful_runs,
            "failed_experiments": failed_runs,
            "total_duration_seconds": total_time,
            "average_duration_seconds": total_time / len(experiment_configs) if len(experiment_configs) > 0 else 0,
            "demo_mode": run_quick_demo,
            "max_epochs": max_epochs_demo if run_quick_demo else max_epochs_full,
            "device": str(device),
            "seed": fixed_seed,
            "execution_method": "pipeline_based_reconstruction_experiment"
        },
        "environment": {
            "python_version": sys.version.split()[0],
            "pytorch_version": torch.__version__,
            "numpy_version": np.__version__,
            "cuda_available": torch.cuda.is_available()
        },
        "pipeline_integration": {
            "experiment_class": "ReconstructionExperiment",
            "runner_function": "run_experiment", 
            "config_approach": "yaml_variants",
            "registry_system": "integrated"
        }
    }
    
    # Add best experiment info if available and results_df exists
    if 'results_df' in locals() and len(results_df) > 0 and "final_val_loss" in results_df.columns:
        best_idx = results_df["final_val_loss"].idxmin()
        best_exp = results_df.loc[best_idx]
        summary["best_experiment"] = {
            "name": best_exp["experiment_name"],
            "validation_loss": float(best_exp["final_val_loss"]),
            "layer_mode": best_exp.get("layer_mode", "N/A"),
            "model_class": best_exp.get("model_class", "N/A")
        }
        
        # Add metrics if available
        metric_cols = [col for col in results_df.columns if col.startswith("metric_")]
        if metric_cols:
            summary["best_experiment"]["metrics"] = {}
            for col in metric_cols:
                metric_name = col.replace("metric_", "")
                if not pd.isna(best_exp[col]):
                    summary["best_experiment"]["metrics"][metric_name] = float(best_exp[col])
    
    # Save summary
    if save_detailed_results:
        summary_file = results_path / "experiment_summary.json"
        with open(summary_file, 'w') as f:
            json.dump(summary, f, indent=2, default=str)
        print(f"💾 Experiment summary saved to: {summary_file}")
    
    # Display final summary
    print(f"\n" + "="*80)
    print(f"🎉 PIPELINE EVALUATION COMPLETED SUCCESSFULLY!")
    print(f"="*80)
    print(f"📊 Total Experiments: {len(experiment_configs)}")
    print(f"✅ Successful: {successful_runs}")
    if failed_runs > 0:
        print(f"❌ Failed: {failed_runs}")
    print(f"⏱️  Total Time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
    print(f"🎯 Pipeline Integration: ReconstructionExperiment ✓ Standard run_experiment ✓")
    
    # Show best performance if results are available
    if 'results_df' in locals() and len(results_df) > 0:
        print(f"\n🏆 Best Performance:")
        print(f"   Experiment: {summary.get('best_experiment', {}).get('name', 'N/A')}")
        print(f"   Validation Loss: {summary.get('best_experiment', {}).get('validation_loss', 'N/A')}")
        
        best_metrics = summary.get('best_experiment', {}).get('metrics', {})
        for metric_name, value in best_metrics.items():
            print(f"   {metric_name.upper()}: {value:.4f}")
    
    print(f"\n📁 Results saved in: {results_dir}")
    print(f"✨ Pipeline-based notebook execution completed!")
        
else:
    print("❌ No experiments were completed successfully")

# Clean up memory
# import gc
# gc.collect()

# print(f"\n🧹 Memory cleanup completed")